# Kev Server — Colab

One-click setup for the Kev decision API: clones `jaredpalmer/kev`, installs the requirements,
downloads `jaredpalmer/kev-0.8b` and starts a FastAPI server on `127.0.0.1:8000`.

**Before you run:** Runtime → Change runtime type → **GPU**. Free Colab's T4 runs `kev-0.8b`
comfortably; on CPU the same server works but takes minutes per request.

Full documentation: https://github.com/AliHaSSan-13/kev-server#running-on-google-colab

In [ ]:
# setup and start the server in the background (first run downloads several GB)
!test -d kev-server || git clone https://github.com/AliHaSSan-13/kev-server.git
!cd kev-server && nohup bash setup.sh > setup.log 2>&1 &

In [ ]:
# wait for the server, then report what it loaded
import time
import urllib.request

BASE_URL = "http://127.0.0.1:8000"

for attempt in range(90):
    try:
        with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as response:
            print(response.read().decode())
            break
    except Exception:
        if attempt % 6 == 0:
            print(f"still setting up ({attempt * 10}s)...", flush=True)
        time.sleep(10)
else:
    print("server did not come up. Last lines of setup.log:\n")
    print(open("/content/kev-server/setup.log").read()[-4000:])

## Expose it with ngrok

The server keeps running in the background — do not restart it. Add your authtoken from
https://dashboard.ngrok.com/get-started/your-authtoken and run the cell.

Prefer the raw command? Open the **Terminal** panel at the bottom left, start a second tab, and
run `ngrok config add-authtoken YOUR_AUTHTOKEN` then `ngrok http 8000`.

In [ ]:
!pip install -q pyngrok

from pyngrok import ngrok, conf

# Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
authtoken = ""
conf.get_default().auth_token = authtoken

# Open HTTP tunnel to port 8000
public_url = ngrok.connect(8000).public_url
print(f"\nPublic API Base URL: {public_url}")

## Ask the model something

`main.py` generated an API key on first run and saved it to `.api_key`, so you don't have to
copy one by hand.

In [ ]:
import json

import requests

API_KEY = open("/content/kev-server/kev/.api_key").read().strip()

response = requests.post(
    f"{public_url}/predict",
    headers={"X-API-Key": API_KEY},
    json={
        "state": "Shoes arrived two weeks late and in the wrong size. I also see two charges on my card.",
        "questions": {
            "department": {
                "type": "choice",
                "instructions": "Which team should handle this?",
                "criteria": {
                    "returns": "Exchanges, refunds, wrong or damaged items",
                    "shipping": "Delivery status, delays, lost packages",
                    "billing": "Charges, invoices, payment problems",
                },
            },
            "escalate": {
                "type": "noul",
                "instructions": "Does this need urgent human attention?",
            },
            "frustration": {
                "type": "score",
                "instructions": "How frustrated is the customer?",
                "criteria": ["Calm", "Frustrated", "Very angry"],
            },
        },
    },
    timeout=180,
)

print(json.dumps(response.json(), indent=2))

---

The ngrok URL is on the public internet: anyone with it *and* the API key can spend your GPU.
Call `ngrok.kill()` when you are done, and Colab closes the tunnel with the runtime anyway.

After the first run the model is cached, so re-running the setup cell on a fresh runtime skips
the download only if the disk survived. Local machines get the same server with
`bash setup.sh` — https://github.com/AliHaSSan-13/kev-server#running-on-a-local-machine